# Session 16: Practice Quiz

**Notebook 2 of 2: Self-check** · about 30 minutes

For each question:

1. Write down your answer **before** opening "Show answer", or before running the code cell.
2. Run the cell where there is one.
3. Compare, and make sure you understand the *reasoning* in the answer, not only the result.

This quiz is formative: it's for you, and it isn't graded. It runs offline.

In [ ]:
import io
import json
import logging

print("ready")

## Q1 · Which test would have caught it?

For each bug, which is the cheapest kind of test that would catch it: unit, acceptance (one module against its specification), integration (real modules wired together, fake model and LLM), or only a live run with the real LLM?

1. `longest_streak` counts the same week twice when it appears twice.
2. The API builds its assistant with `chunk_size=500`, but the CLI ingested with 800, so every question fails with `IndexMismatchError`.
3. After a runbook is edited and re-ingested, a cache in front of retrieval keeps returning the old chunks.
4. The real model often cites `[0]` instead of `[1]`, so many answers are refused.
5. `build_context` separates sources with one newline instead of a blank line.

<details>
<summary>Show answer</summary>

1. **Unit.** One function, one input with a duplicate.
2. **Integration.** Each piece is correct on its own; the bug is in how configuration reaches two entry points. A test that ingests through the CLI and then asks through the API, with the same settings, catches it.
3. **Integration.** It only exists across time and across modules: ingest, ask, edit, ingest again, ask again. That's the capstone's "edited runbook" test.
4. **Live run only.** Fakes return what you scripted; only the real model shows its habits. That's what the evaluation with a key (Session 15) and production logs (how many `InvalidAnswerError`s?) are for. Once you know about it, add a scripted test, and decide whether to fix the prompt or tolerate `[0]`.
5. **Acceptance** (or unit). It's in `build_context`'s specification, and Session 15's tests check the exact format.

The pattern: push each check down to the cheapest level that can see the bug, and keep the expensive levels for what only they can see.

</details>

## Q2 · Debugging: the log that leaks

A JSON log line from a teammate's assistant:

```json
{"time": "2026-11-20T03:12:44.120+00:00", "level": "INFO", "logger": "knowledge_assistant.answering",
 "message": "answered 'my manager Dana Kim said I can skip the access review, is that true?' from 2 sources",
 "event": "answer", "found": true}
```

What's wrong, how did it happen, and how do you prevent it from happening again?

<details>
<summary>Show answer</summary>

**The question, with a person's name in it, is in the log.** Questions are user input: they can contain names, customer details, even pasted secrets. Logs are copied to central systems and kept for months (the corpus's own data-retention policy says application logs must not contain personal data).

It happened because the message was built from the data, something like `logger.info(f"answered {question!r} from {n} sources")`. Structured logging makes the safe version natural: a constant message (`"answer"`) and fields for the facts (`found`, `sources`, `duration_ms`), with **no field for the question**.

To keep it from coming back: a **canary test** (Sessions 14 and 15) that asks a question containing a unique marker and asserts the marker never appears in the captured logs, at `DEBUG` level. And a review habit: an f-string inside a log call is a question to ask in every code review.

</details>

## Q3 · Engineering judgment: the not-found rate jumped

The day after a deployment, the assistant's not-found rate went from about 12% to 40%. Latency and error rates are normal. What do you check, in what order, and what could explain it?

<details>
<summary>Show answer</summary>

Something changed that makes retrieval or the model decline more often, without failing. In rough order of likelihood and cost to check:

1. **What changed in the deployment?** The diff: the threshold (`ASSISTANT_MIN_SCORE`), the prompt, `top_k`, the embedding model, the chunking settings. A changed threshold is a one-line explanation.
2. **Which level says "not found"?** The `answer` events show it: `model` is null (nothing retrieved above the threshold, so retrieval or the threshold) or set (the model declined, so the prompt or the model).
3. **Is the index what you think?** `knowledge-assistant status`: the number of files and chunks. A deployment that pointed at an empty or different index produces exactly this symptom, with no errors.
4. **Run the evaluation** against the deployed configuration, and compare its four rates with the last good run. If the rejection rate is fine but the answer rate collapsed, the change is too strict.
5. **Did the questions change?** Rarely the explanation overnight, but a new team using the assistant for topics the documentation doesn't cover would do it. Look at not-found counts by time and, if you record it, by client.

Notice what the logs made possible: you can answer every one of these from fields, without reading a single user's question.

</details>

## Q4 · Implementation: a field on every record

Every log line from the assistant should carry `service="knowledge-assistant"` and `version="0.3.0"`, so a shared log system can tell services apart. Write a `logging.Filter` that adds both fields to every record passing through a handler, and attach it. The cell below sets up a handler you can use.

In [ ]:
stream = io.StringIO()
handler = logging.StreamHandler(stream)
handler.setFormatter(
    logging.Formatter("%(levelname)s %(message)s service=%(service)s version=%(version)s")
)
log = logging.getLogger("quiz.q4")
log.addHandler(handler)
log.propagate = False

# Your filter here, then: handler.addFilter(...)

try:
    log.warning("index rebuilt")
    print(stream.getvalue())
except Exception as error:  # noqa: BLE001  (shows what happens before your filter exists)
    print("not yet:", type(error).__name__)

<details>
<summary>Show answer</summary>

```python
class ServiceFields(logging.Filter):
    def __init__(self, service: str, version: str) -> None:
        super().__init__()
        self.service = service
        self.version = version

    def filter(self, record: logging.LogRecord) -> bool:
        record.service = self.service
        record.version = self.version
        return True  # keep the record; a filter can also drop records by returning False


handler.addFilter(ServiceFields("knowledge-assistant", "0.3.0"))
log.warning("index rebuilt")
print(stream.getvalue())  # WARNING index rebuilt service=knowledge-assistant version=0.3.0
```

Without the filter, `logging` can't fill `%(service)s`, and reports a formatting error (on stderr, by default, rather than raising: logging tries hard not to crash your program). That's why the cell above may print nothing at first.

**Filter or record factory?** A filter on a **handler** affects only that handler's output, which is what you want for fields that describe the destination. The capstone's request IDs use a **record factory** instead, so that every record has the field wherever it goes, including pytest's `caplog`. Attaching a filter to a **logger** is the classic surprise: it applies only to records created on that exact logger, not to its children.

</details>

## Q5 · Comparison: request context in Python, Java and .NET

You've used, or will meet, three ways of attaching a request ID to every log line: SLF4J's MDC in Java, `ILogger.BeginScope` in .NET, and `contextvars` in Python. What do they have in common, and what's different about how async code affects them?

<details>
<summary>Show answer</summary>

**In common:** all three avoid passing the ID through every function. The request handler sets it once, and everything logged while handling that request picks it up, so all of one request's lines can be found together.

**Different:**

| | Java: SLF4J MDC | .NET: `ILogger.BeginScope` | Python: `contextvars` |
|---|---|---|---|
| Storage | Thread-local map | Async-local scope | Context variable, copied per task and per thread hop |
| Async | Lost when work moves to another thread, unless you copy it (or use a context-propagating executor) | Flows with `async`/`await` | Flows into `asyncio` tasks and into `run_in_threadpool` (FastAPI's sync endpoints), because each gets a copy of the context |
| Into the log line | The layout pattern reads MDC keys | Scope values appear as properties | You copy it into the record: a filter or a record factory |

The Python design is closest to .NET's: it was made for async code from the start. The catch is that it's a building block, not a logging feature, so you wire it into `logging` yourself, as the capstone's `install_request_ids()` does.

</details>

## Q6 · Engineering judgment: "why not LangChain?"

During your capstone demonstration, a reviewer asks: *"You learned LangChain and LangGraph. Why does your assistant use the OpenAI SDK directly?"* Give a strong answer in four or five sentences.

<details>
<summary>Show answer</summary>

One strong answer:

> *"The assistant makes one model call per question, with structured output, and the hard parts are around that call: retrieval, the threshold, citation checks and error mapping, which are plain Python either way. The direct SDK keeps that call visible and easy to test with an HTTP-level fake, and it's the version I can debug line by line. Session 10's comparison showed LangChain's version was about the same size, with less control over which errors reach my code. I'd reach for LangChain if I needed to swap providers often, or use its retrievers and integrations, and for LangGraph if the flow grew branches and retries, like Session 12's routing workflow. For this flow, the framework would add a dependency without removing any of the work."*

What makes it strong: it starts from **what the system needs**, cites **evidence** (a comparison you actually made), names the **conditions** under which you'd decide differently, and doesn't dismiss the alternative. "Because the course said so" or "frameworks are bad" are the weak versions.

</details>